# Reto 1: Consulta y análisis de datos meteorológicos

## 11. Guarda los resultados

Guardaremos los registros meteorológicos en formatos `JSON` y `CSV`. Después volveremos a leer los archivos para comprobar que coinciden con los datos originales.

## Preparación de los datos

Obtenemos las fechas y temperaturas de enero de 2025. Cada registro incluirá la temperatura media, máxima y mínima, además de la amplitud térmica diaria (`máxima - mínima`).

In [ ]:
import csv
import json
from pathlib import Path

import requests

url = "https://archive-api.open-meteo.com/v1/archive"
parametros = {
    "latitude": 37.7796,
    "longitude": -3.7849,
    "start_date": "2025-01-01",
    "end_date": "2025-01-31",
    "daily": "temperature_2m_mean,temperature_2m_max,temperature_2m_min",
    "timezone": "Europe/Madrid",
}

respuesta = requests.get(url, params=parametros, timeout=30)
respuesta.raise_for_status()
datos = respuesta.json()

diarios = datos["daily"]
fechas = diarios["time"]
temperaturas_medias = diarios["temperature_2m_mean"]
temperaturas_maximas = diarios["temperature_2m_max"]
temperaturas_minimas = diarios["temperature_2m_min"]

registros = [
    {
        "fecha": fecha,
        "media": media,
        "maxima": maxima,
        "minima": minima,
        "amplitud_termica": round(maxima - minima, 1),
    }
    for fecha, media, maxima, minima in zip(
        fechas,
        temperaturas_medias,
        temperaturas_maximas,
        temperaturas_minimas,
    )
]

directorio_salida = Path("Reto1/resultados") if Path("Reto1").exists() else Path("resultados")
directorio_salida.mkdir(parents=True, exist_ok=True)

print("Número de registros:", len(registros))
print("Primer registro:", registros[0])
print("Directorio de salida:", directorio_salida.resolve())

## 1. Guarda los registros en JSON

El nombre del archivo identifica la ciudad y el periodo analizado. `ensure_ascii=False` conserva correctamente los caracteres especiales e `indent=4` hace que el contenido sea más legible.

In [ ]:
ruta_json = directorio_salida / "temperaturas_jaen_2025-01.json"

with ruta_json.open("w", encoding="utf-8") as archivo_json:
    json.dump(registros, archivo_json, ensure_ascii=False, indent=4)

print("JSON guardado en:", ruta_json)

## 2. Guarda los registros en CSV

El archivo contiene una fila por día y una columna para cada variable, incluida la amplitud térmica. `newline=""` evita que aparezcan líneas vacías adicionales al escribir el CSV en Windows.

In [ ]:
ruta_csv = directorio_salida / "temperaturas_jaen_2025-01.csv"
columnas = ["fecha", "media", "maxima", "minima", "amplitud_termica"]

with ruta_csv.open("w", encoding="utf-8", newline="") as archivo_csv:
    escritor = csv.DictWriter(archivo_csv, fieldnames=columnas)
    escritor.writeheader()
    escritor.writerows(registros)

print("CSV completo guardado en:", ruta_csv)

## 3. Crea un CSV con la fecha y la temperatura media

Creamos una versión reducida que contiene solamente las dos columnas solicitadas.

In [ ]:
registros_medias = [
    {"fecha": registro["fecha"], "media": registro["media"]}
    for registro in registros
]

ruta_csv_medias = directorio_salida / "temperaturas_medias_jaen_2025-01.csv"

with ruta_csv_medias.open("w", encoding="utf-8", newline="") as archivo_csv:
    escritor = csv.DictWriter(archivo_csv, fieldnames=["fecha", "media"])
    escritor.writeheader()
    escritor.writerows(registros_medias)

print("CSV de medias guardado en:", ruta_csv_medias)

## 4. Vuelve a leer los archivos y comprueba los datos

Al leer un CSV, todos los campos se obtienen inicialmente como texto. Por eso convertimos las columnas numéricas a `float` antes de compararlas con los registros originales.

In [ ]:
with ruta_json.open("r", encoding="utf-8") as archivo_json:
    registros_desde_json = json.load(archivo_json)

with ruta_csv.open("r", encoding="utf-8", newline="") as archivo_csv:
    lector = csv.DictReader(archivo_csv)
    registros_desde_csv = [
        {
            "fecha": fila["fecha"],
            "media": float(fila["media"]),
            "maxima": float(fila["maxima"]),
            "minima": float(fila["minima"]),
            "amplitud_termica": float(fila["amplitud_termica"]),
        }
        for fila in lector
    ]

with ruta_csv_medias.open("r", encoding="utf-8", newline="") as archivo_csv:
    lector = csv.DictReader(archivo_csv)
    medias_desde_csv = [
        {"fecha": fila["fecha"], "media": float(fila["media"])}
        for fila in lector
    ]

json_coincide = registros_desde_json == registros
csv_coincide = registros_desde_csv == registros
csv_medias_coincide = medias_desde_csv == registros_medias

print("El JSON coincide con los datos originales:", json_coincide)
print("El CSV completo coincide con los datos originales:", csv_coincide)
print("El CSV de medias coincide con los datos originales:", csv_medias_coincide)

assert json_coincide and csv_coincide and csv_medias_coincide
print("Comprobación completada: los tres archivos conservan los datos esperados.")

## Archivos generados

Al ejecutar el notebook se crean estos archivos dentro de la carpeta `resultados`:

- `temperaturas_jaen_2025-01.json`: todos los registros en JSON.
- `temperaturas_jaen_2025-01.csv`: todos los registros en formato tabular.
- `temperaturas_medias_jaen_2025-01.csv`: únicamente la fecha y la temperatura media.

Los nombres indican la variable, la ciudad y el periodo, lo que facilita identificar y ordenar los resultados.

## Comprueba tu comprensión

### 1. ¿Qué ventajas tienen JSON y CSV?

**Respuesta:**

| Formato | Ventajas | Limitaciones |
|---|---|---|
| `JSON` | Conserva números, textos, listas y diccionarios; es adecuado para datos con una estructura compleja y para intercambiar información entre programas. | Es más extenso y menos cómodo para trabajar directamente en una hoja de cálculo. |
| `CSV` | Es sencillo, compacto y fácil de abrir con hojas de cálculo o programas de análisis. Resulta apropiado para datos organizados en filas y columnas. | No conserva los tipos de datos al leerlo y no representa bien estructuras anidadas. |

### 2. ¿Por qué se vuelven a leer y comparar los archivos?

**Respuesta:**

Guardar un archivo sin errores no garantiza por sí solo que su contenido sea correcto. Al volver a leerlo y compararlo comprobamos que no se han perdido registros, columnas ni valores durante la escritura.

### 3. ¿Qué aporta la columna `amplitud_termica`?

**Respuesta:**

Muestra la diferencia entre la temperatura máxima y la mínima de cada día. Permite estudiar cuánto varió la temperatura dentro de una misma jornada sin tener que repetir el cálculo al leer el archivo.

## Conclusión

Los datos quedan guardados en dos formatos complementarios. JSON conserva directamente la estructura y los tipos de los registros, mientras que CSV facilita su consulta como tabla. La lectura posterior y las comparaciones verifican que los archivos generados contienen la misma información que los datos originales.